# Quantum Threat to AES: Grover Key Search Demo

**Mini project goal:** demonstrate whether a quantum computer can search an AES key faster than classical brute force.

We cannot simulate full AES-128 as a reversible quantum circuit on a laptop, so this notebook uses:
- real AES encryption for a tiny classical brute-force baseline,
- a toy Grover oracle for a 2–5 bit keyspace,
- success-probability and noise analysis,
- mathematical extrapolation to AES-128 and AES-256.

In [ ]:
# Install once if needed:
# !pip install qiskit qiskit-aer pycryptodome matplotlib numpy pandas

import math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Crypto.Cipher import AES
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, pauli_error
from qiskit.visualization import plot_histogram

## 1. Classical AES brute-force baseline

We keep AES itself real, but shrink the searched keyspace by varying only a few low-order key values and padding the rest of the 16-byte AES key with zero bytes.

In [ ]:
def padded_aes_key(candidate: int) -> bytes:
    return bytes([candidate]).ljust(16, b"\x00")

def classical_bruteforce_aes(key_bits: int, target_key: int, plaintext: bytes):
    cipher = AES.new(padded_aes_key(target_key), AES.MODE_ECB)
    ciphertext = cipher.encrypt(plaintext)

    start = time.perf_counter()
    for attempts, candidate in enumerate(range(2**key_bits), start=1):
        trial = AES.new(padded_aes_key(candidate), AES.MODE_ECB)
        if trial.encrypt(plaintext) == ciphertext:
            return candidate, attempts, time.perf_counter() - start

plaintext = b"Hello, World!!!!"  # exactly 16 bytes
rows = []
for bits in range(1, 9):
    target = min(5, 2**bits - 1)
    found, attempts, seconds = classical_bruteforce_aes(bits, target, plaintext)
    rows.append({"key_bits": bits, "keyspace": 2**bits, "target_key": target, "found_key": found, "attempts": attempts, "time_sec": seconds})

classical_df = pd.DataFrame(rows)
classical_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(classical_df["key_bits"], classical_df["attempts"], marker="o")
plt.xlabel("Toy key size (bits)")
plt.ylabel("Attempts")
plt.title("Classical brute force grows linearly with keyspace")
plt.grid(True)
plt.show()

## 2. Grover circuit utilities

A Grover circuit has three parts:
1. Hadamard layer: creates superposition.
2. Oracle: phase-flips the correct key.
3. Diffusion operator: amplifies the correct key's amplitude.

In this project, the oracle is simplified: it directly marks a target state. In a full AES attack, the oracle would have to implement AES reversibly.

In [ ]:
def optimal_grover_iterations(n_qubits):
    return max(1, round((math.pi / 4) * math.sqrt(2**n_qubits)))

def apply_phase_oracle(qc, target, n_qubits):
    bits_le = format(target, f"0{n_qubits}b")[::-1]  # qubit 0 is least significant
    for i, bit in enumerate(bits_le):
        if bit == "0":
            qc.x(i)

    if n_qubits == 1:
        qc.z(0)
    else:
        qc.h(n_qubits - 1)
        qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
        qc.h(n_qubits - 1)

    for i, bit in enumerate(bits_le):
        if bit == "0":
            qc.x(i)

def apply_diffuser(qc, n_qubits):
    qc.h(range(n_qubits))
    qc.x(range(n_qubits))
    if n_qubits == 1:
        qc.z(0)
    else:
        qc.h(n_qubits - 1)
        qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
        qc.h(n_qubits - 1)
    qc.x(range(n_qubits))
    qc.h(range(n_qubits))

def build_grover_circuit(n_qubits, target, iterations=None):
    if iterations is None:
        iterations = optimal_grover_iterations(n_qubits)
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.h(range(n_qubits))
    qc.barrier(label="superposition")
    for _ in range(iterations):
        apply_phase_oracle(qc, target, n_qubits)
        qc.barrier(label="oracle")
        apply_diffuser(qc, n_qubits)
        qc.barrier(label="diffusion")
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def run_counts(qc, shots=1024, noise_model=None):
    simulator = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, simulator)
    return simulator.run(tqc, shots=shots).result().get_counts()

def success_probability(counts, target, n_qubits):
    target_string = format(target, f"0{n_qubits}b")
    return counts.get(target_string, 0) / sum(counts.values())

## 3. Grover correctness demo

In [ ]:
n_qubits = 3
correct_key = 5  # binary 101
iterations = optimal_grover_iterations(n_qubits)

qc = build_grover_circuit(n_qubits, correct_key, iterations)
print(qc.draw(output="text"))

counts = run_counts(qc, shots=1024)
print(counts)
print("Success probability:", success_probability(counts, correct_key, n_qubits))
plot_histogram(counts)

## 4. Success probability vs number of Grover iterations

Grover must be stopped near the optimal point. If we keep applying iterations, the probability rotates past the target and drops again.

In [ ]:
rows = []
for r in range(0, 8):
    qc_r = build_grover_circuit(n_qubits, correct_key, r)
    counts_r = run_counts(qc_r, shots=1024)
    rows.append({"iterations": r, "success_probability": success_probability(counts_r, correct_key, n_qubits)})

iter_df = pd.DataFrame(rows)
iter_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(iter_df["iterations"], iter_df["success_probability"], marker="o")
plt.axvline((math.pi/4)*math.sqrt(2**n_qubits), linestyle="--", label="π/4 √N")
plt.xlabel("Grover iterations")
plt.ylabel("Success probability")
plt.ylim(0, 1.05)
plt.title("Grover's algorithm has an optimal stopping point")
plt.legend()
plt.grid(True)
plt.show()

## 5. Classical vs Grover complexity extrapolation

In [ ]:
key_sizes = np.arange(1, 257)
classical_log2 = key_sizes          # log2(2^n)
grover_log2 = key_sizes / 2         # log2(2^(n/2))

plt.figure(figsize=(10, 6))
plt.plot(key_sizes, classical_log2, label="Classical brute force: 2^n")
plt.plot(key_sizes, grover_log2, label="Grover search: 2^(n/2)")
plt.axvline(128, linestyle="--", label="AES-128")
plt.axvline(256, linestyle="--", label="AES-256")
plt.xlabel("Key size (bits)")
plt.ylabel("log2 operations")
plt.title("Classical vs Quantum Exhaustive Key Search")
plt.legend()
plt.grid(True)
plt.show()

print("AES-128 classical: 2^128, Grover idealized: about 2^64 oracle calls")
print("AES-256 classical: 2^256, Grover idealized: about 2^128 oracle calls")

## 6. Noise analysis

We compare three simple noise channels: depolarizing, bit-flip, and phase-flip. This makes the project more impressive because it shows that real quantum hardware errors reduce Grover's success probability.

In [ ]:
def make_noise_model(kind, p):
    noise_model = NoiseModel()
    if kind == "depolarizing":
        err1 = depolarizing_error(p, 1)
        err2 = depolarizing_error(p, 2)
    elif kind == "bit_flip":
        err1 = pauli_error([("X", p), ("I", 1 - p)])
        err2 = err1.tensor(err1)
    elif kind == "phase_flip":
        err1 = pauli_error([("Z", p), ("I", 1 - p)])
        err2 = err1.tensor(err1)
    else:
        raise ValueError("unknown noise kind")

    noise_model.add_all_qubit_quantum_error(err1, ["h", "x", "z", "sx", "rz"])
    noise_model.add_all_qubit_quantum_error(err2, ["cx", "cz"])
    return noise_model

error_rates = np.linspace(0, 0.15, 10)
noise_kinds = ["depolarizing", "bit_flip", "phase_flip"]
noise_rows = []

for kind in noise_kinds:
    for p in error_rates:
        nm = make_noise_model(kind, float(p))
        qc_noise = build_grover_circuit(n_qubits, correct_key, iterations)
        counts_noise = run_counts(qc_noise, shots=1024, noise_model=nm)
        noise_rows.append({"noise_type": kind, "error_rate": p, "success_probability": success_probability(counts_noise, correct_key, n_qubits)})

noise_df = pd.DataFrame(noise_rows)
noise_df.head()

In [ ]:
plt.figure(figsize=(9, 5))
for kind in noise_kinds:
    part = noise_df[noise_df["noise_type"] == kind]
    plt.plot(part["error_rate"], part["success_probability"], marker="o", label=kind)
plt.xlabel("Error rate per gate")
plt.ylabel("Success probability")
plt.ylim(0, 1.05)
plt.title("Grover's Success Probability Under Noise")
plt.legend()
plt.grid(True)
plt.show()

## 7. Final conclusion

This project demonstrates Grover's algorithm on a reduced keyspace. The circuit proves amplitude amplification: the correct key obtains much higher measurement probability than wrong keys. Under noise, the success probability decreases, showing why practical quantum cryptanalysis requires fault-tolerant quantum computers.

For full AES, the key idea is mathematical extrapolation: a keyspace of size N classically requires O(N) brute-force trials, while Grover requires O(√N) oracle calls. Therefore, idealized Grover search changes AES-128 exhaustive key search from about 2^128 to about 2^64 oracle calls, and AES-256 from about 2^256 to about 2^128 oracle calls.

Important limitation: the implemented oracle is simplified. A real AES oracle must reversibly compute AES encryption, compare against ciphertext, mark the matching key, and uncompute all temporary states.